## INSTALL DEPENDENCIES

In [ ]:
# !pip install git+https://github.com/huggingface/transformers.git qwen_vl_utils torchvision accelerate>=0.26.0
# pip install git+https://github.com/huggingface/transformers accelerate bitsandbytes qwen_vl_utils torchvision
!pip install 'torch>=2.6' 'transformers>=4.49' 'accelerate>=0.26.0' bitsandbytes qwen_vl_utils torchvision

In [2]:
!nvidia-smi

Mon Jul 14 08:58:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:46:00.0 Off |                    0 |
| N/A   35C    P0             82W /  700W |       1MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## IMPORT FUNCTIONS

In [ ]:
from simulations_core import *

## DEFINE PARAMETERS

In [ ]:
family = 'qwen' #'gemma', 'mistral'
model_label = 'qwen_32B' # 'qwen_3B', 'qwen_7B', 'qwen_32B', 'qwen2_2B', 'qwen2_7B', 'gemma_4B', 'gemma_12B', 'gemma_27B', 'mistral_24B'
task = 'color_recognition' # 'asch_lines', 'dots_estimation', 'color_recognition'
n_simulations = 32

labels = ['A', 'B']

# QWEN

if model_label == 'qwen_3B':
    model, processor = import_qwen_3B()
if model_label == 'qwen_7B':
    model, processor = import_qwen_7B()
if model_label == 'qwen_32B':
    model, processor = import_qwen_32B()
if model_label == 'qwen2_2B':
    model, processor = import_qwen2_2B()
if model_label == 'qwen2_7B':
    model, processor = import_qwen2_7B()

if family == 'qwen':
    generation_kwargs = {
        "max_new_tokens": 256,
        "do_sample": True,
        "top_k": 50,
        "top_p": 0.95,
        "temperature": 0.7,
        "output_scores": True,               # <--- Aggiunto
        "return_dict_in_generate": True,      # <--- Aggiunto
        "pad_token_id": processor.tokenizer.eos_token_id  # 👈 aggiungi questa riga
    }

# GEMMA

if model_label == 'gemma_4B':
    model, processor = import_gemma_4B()
if model_label == 'gemma_12B':
    model, processor = import_gemma_12B()
if model_label == 'gemma_27B':
    model, processor = import_gemma_27B()


if family == 'gemma':
    generation_kwargs = {
        "max_new_tokens": 256,
        "do_sample": False,
        # "top_p": 0.9,
        # "temperature": 0.7,
        "return_dict_in_generate": True, 
        "output_scores": True,               # <--- Aggiunto
        "use_cache" : False  # 👈 aggiungi questa riga
    }

# MISTRAL

if family == 'mistral':
    model, processor = import_mistral_24B()
    generation_kwargs = {
        "max_new_tokens": 256,
        "do_sample": True,
        "top_k": 50,
        "top_p": 0.95,
        "temperature": 0.15,
        "output_scores": True,               # <--- Aggiunto
        "return_dict_in_generate": True,      # <--- Aggiunto
        "pad_token_id": processor.tokenizer.eos_token_id  # 👈 aggiungi questa riga
    }

Loading checkpoint shards:   0%|          | 0/18 [00:00<?, ?it/s]

## DEFINE FOLDERS

In [ ]:
if task == 'asch_lines':
    folder_general = f'images/images_lines_{model_label}'
    folder_ambiguity = f'images/images_lines_perplexity_diff_{model_label}'
if task == 'color_recognition':
    folder_general = f'images/images_color_{model_label}'
    folder_ambiguity = f'images/images_color_perplexity_bound_{model_label}'
if task == 'dots_estimation':
    folder_general = f'images/images_dots_{model_label}'
    folder_ambiguity = f'images/images_dots_perplexity_diff_{model_label}'

## DEFINE PROMPT

In [ ]:
def get_base_prompt(task, labels):
    if task == 'asch_lines':
        prompt_base = (
            f"In the image, there are three vertical lines labeled {labels[0]}, REFERENCE LINE, and {labels[1]}.\n"
            f"Which of the lines, {labels[0]} or {labels[1]}, is the same length as the REFERENCE LINE?\n"
        )
    if task == 'color_recognition':
        prompt_base = (
            f"In the image, there are three colored squares labeled {labels[0]}, REFERENCE COLOR, and {labels[1]}.\n"
            f"Which of the squares, {labels[0]} or {labels[1]}, has the same color as the REFERENCE COLOR?\n"
        )
    if task == 'dots_estimation':
        prompt_base = (
            f"In the image, there are three boxes labeled {labels[0]}, REFERENCE BOX, and {labels[1]}.\n"
            f"Which of the boxes, {labels[0]} or {labels[1]}, contains the same number of black dots as the REFERENCE BOX?\n"
        )

    return prompt_base

prompt_base = get_base_prompt(task, labels)

## MAJORITY EFFECT

In [ ]:
import string
import os
import pickle
import re

labels = ['A', 'B']
list_N_wrongs = range(0, 11, 1)

folder = folder_general

conditions = ['random', 'right', 'wrong', 'minority', 'two_allies']

# Crea la directory "data" se non esiste
os.makedirs(f'data/data_{task}_{model_label}', exist_ok=True)

for condition in conditions:
    probs_all_N = []
    print(f'condition = {condition}')

    for correct_answer in ['A', 'B']:
        pattern = re.compile(rf"_({correct_answer})_p=.*\.png$")
        all_images = [f for f in os.listdir(folder) if pattern.search(f)]
        if not all_images:
            continue
        
        probs_N = []

        for N_wrong in list_N_wrongs:
            print(f'N_wrong = {N_wrong}')
            probs_wrong = []

            while len(probs_wrong) < n_simulations:

                # Genera le risposte dei confederati
                list_responses = []
                if N_wrong > 0:
                    for _ in range(N_wrong):
                        if condition == 'random':
                            label = random.choice(labels)
                        elif condition == 'right':
                            label = correct_answer
                        elif condition in ['wrong', 'minority', 'two_allies']:
                            label = labels[1 - labels.index(correct_answer)]
                        response = get_neutral_responses(label, 1)[0]
                        list_responses.append(response)
                
                    if condition == 'minority':
                        # Aggiungi UNA risposta giusta in posizione random
                        correct_response = get_neutral_responses(correct_answer, 1)[0]
                        insert_idx = random.randint(0, len(list_responses))
                        list_responses.insert(insert_idx, correct_response)
                
                    elif condition == 'two_allies':
                        # Aggiungi DUE risposte giuste in posizioni random
                        correct_responses = get_neutral_responses(correct_answer, 2)
                        insert_idxs = sorted(random.sample(range(len(list_responses) + 2), 2))
                        for idx, resp in zip(insert_idxs, correct_responses):
                            list_responses.insert(idx, resp)

                image = random.choice(all_images)
                image_path = os.path.join(folder, image)

                # Crea il prompt
                if N_wrong == 0:
                    prompt = (
                        f"{prompt_base}"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )
                else:
                    prompt = (
                        f"{prompt_base}"
                        "The participants before you provided the following replies:\n"
                        + "\n".join(f"reply {j+1}: {r}" for j, r in enumerate(list_responses)) + "\n"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )

                #inference
                if family == 'qwen':
                    probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'gemma':
                    probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'mistral':
                    probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
    
                probs_wrong.append(1 - probs[correct_answer])

                ##########################

            probs_N.append(probs_wrong)

        probs_all_N.append(probs_N)

    # Aggrega tutti i dati
    probs_all_N = np.array(probs_all_N)
    probs_all_N = probs_all_N.transpose(1, 0, 2).reshape(len(list_N_wrongs), -1)

    # 🔥 Salva su file pickle
    save_path = f'data/data_{task}_{model_label}/majority_{condition}.pkl'
    with open(save_path, 'wb') as f:
        pickle.dump(probs_all_N, f)
    print(f'Saved {save_path}')

## TASK DIFFICULTY AND PERFORMANCE

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from PIL import Image
from collections import defaultdict

# === PARAMETRI ===
labels = ['A', 'B']
list_N_wrongs = list(range(0, 11))
n_simulations_here = 1

# === CARTELLE ===
folder = folder_ambiguity
output_folder = f'data/data_{task}_{model_label}'
os.makedirs(output_folder, exist_ok=True)

# === IMMAGINI ===
all_images = [f for f in os.listdir(folder) if f.endswith('.png')]
results = []

for idx, image in enumerate(all_images):
    print(f"🔍 Processing image {idx+1}/{len(all_images)}: {image}")

    # Estrai bound, logit e correct answer
    match_bound = re.search(r"bound=(\d+)\.png", image)
    match_logit = re.search(r"logit=(\d+\.\d+)", image)
    match_answer = re.search(r"_([AB])_p=", image)

    if not (match_bound and match_logit and match_answer):
        print(f"❌ Skipping: couldn't parse metadata from filename.")
        continue

    bound = int(match_bound.group(1))
    logit = float(match_logit.group(1))
    correct_answer = match_answer.group(1)
    image_path = Image.open(os.path.join(folder, image))

    probs_wrong_total = []

    for N_wrong in list_N_wrongs:
        probs_wrong = []

        while len(probs_wrong) < n_simulations_here:
            list_responses = get_neutral_responses('B' if correct_answer == 'A' else 'A', N_wrong)
            
            if N_wrong == 0:
                prompt = (
                    f"{prompt_base}"
                    f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                )
            else:
                prompt = (
                    f"{prompt_base}"
                    "The participants before you provided the following replies:\n"
                    + "\n".join(f"reply {j+1}: {r}" for j, r in enumerate(list_responses)) + "\n"
                    f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                )

            #inference
            if family == 'qwen':
                probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
            if family == 'gemma':
                probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
            if family == 'mistral':
                probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)

            probs_wrong.append(1 - probs[correct_answer])

        probs_wrong_total.append(probs_wrong)

    # === CALCOLO AUC ===
    probs_wrong_mean = np.mean(probs_wrong_total, axis=1)
    auc = np.trapz(probs_wrong_mean, list_N_wrongs) / (list_N_wrongs[-1] - list_N_wrongs[0])

    # === SALVA RISULTATO ===
    results.append({
        "bound": bound,
        "logit": logit,
        "AUC": auc
    })

# === SALVA CSV ===
df = pd.DataFrame(results)
output_path = os.path.join(output_folder, "all_images_auc_logit.csv")
df.to_csv(output_path, index=False)
print(f"\n💾 Salvato: {output_path}")

## SOCIAL IMPACT THEORY (STRENGTH)

In [ ]:
import string
import os
import pickle
import re

labels = ['A', 'B']
list_N_wrongs = range(0, 11, 1)

folder = folder_general

individuals = ['chatbots', 'kids', 'strangers', 'people', 'humans', 'experts', 'policemen', 'scientists', 'doctors', 'judges', 'military officers']

# Crea la directory "data" se non esiste
os.makedirs(f'data/data_{task}_{model_label}', exist_ok=True)

for individual in individuals:
    probs_all_N = []
    print(f'individual = {individual}')

    for correct_answer in ['A', 'B']:
        pattern = re.compile(rf"_({correct_answer})_p=.*\.png$")
        all_images = [f for f in os.listdir(folder) if pattern.search(f)]
        if not all_images:
            continue
        
        probs_N = []

        for N_wrong in list_N_wrongs:
            print(f'N_wrong = {N_wrong}')
            probs_wrong = []

            while len(probs_wrong) < n_simulations:
                image = random.choice(all_images)
                image_path = os.path.join(folder, image)

                # Genera le risposte dei confederati
                list_responses = []
                if N_wrong > 0:
                    for _ in range(N_wrong):

                        label = labels[1 - labels.index(correct_answer)]
                        response = get_neutral_responses(label, 1)[0]
                        list_responses.append(response)


                # Crea il prompt
                if N_wrong == 0:
                    prompt = (
                        f"{prompt_base}"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )
                else:
                    prompt = (
                        f"{prompt_base}"
                        f"The {individual} before you provided the following replies:\n"
                        + "\n".join(f"reply {j+1}: {r}" for j, r in enumerate(list_responses)) + "\n"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )

                #inference
                if family == 'qwen':
                    probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'gemma':
                    probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'mistral':
                    probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
    
                probs_wrong.append(1 - probs[correct_answer])

            probs_N.append(probs_wrong)

        probs_all_N.append(probs_N)

    # Aggrega tutti i dati
    probs_all_N = np.array(probs_all_N)
    probs_all_N = probs_all_N.transpose(1, 0, 2).reshape(len(list_N_wrongs), -1)

    # 🔥 Salva su file pickle
    save_path = f'data/data_{task}_{model_label}/majority_{individual}.pkl'
    with open(save_path, 'wb') as f:
        pickle.dump(probs_all_N, f)
    print(f'Saved {save_path}')

## SOCIAL IMPACT THEORY (IMMEDIACY) ETHNICITY

In [ ]:
import string
import os
import pickle
import re

labels = ['A', 'B']
list_N_wrongs = range(0, 11, 1)

folder = folder_general

ethnicities = ["European","African","Asian","Hispanic"]

conditions = ['same_ethnicity', 'different_ethnicity']

# Crea la directory "data" se non esiste
os.makedirs(f'data/data_{task}_{model_label}', exist_ok=True)

for condition in conditions:
    probs_all_N = []
    print(f'condition = {condition}')

    for correct_answer in ['A', 'B']:
        pattern = re.compile(rf"_({correct_answer})_p=.*\.png$")
        all_images = [f for f in os.listdir(folder) if pattern.search(f)]
        if not all_images:
            continue
        
        probs_N = []

        for N_wrong in list_N_wrongs:
            print(f'N_wrong = {N_wrong}')
            probs_wrong = []

            while len(probs_wrong) < n_simulations:
                image = random.choice(all_images)
                image_path = os.path.join(folder, image)

                # Genera le risposte dei confederati
                list_responses = []
                if N_wrong > 0:
                    for _ in range(N_wrong):
                        label = labels[1 - labels.index(correct_answer)]
                        response = get_neutral_responses(label, 1)[0]
                        list_responses.append(response)


                if condition == 'same_ethnicity':
                    ethnicity1 = random.choice(ethnicities)
                    ethnicity2 = ethnicity1
                else:
                    ethnicity1, ethnicity2 = random.sample(ethnicities, 2)
            
                # Crea il prompt
                if N_wrong == 0:
                    prompt = (
                        f"{prompt_base}"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )
                else:
                    prompt = (
                        f"Your ethnicity is {ethnicity1}.\n"
                        f"{prompt_base}"
                        f"The participants before you provided the following replies:\n"
                        + "\n".join(f"{ethnicity2}: reply {j+1}: {r}" for j, r in enumerate(list_responses)) + "\n"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )

                #inference
                if family == 'qwen':
                    probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'gemma':
                    probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'mistral':
                    probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
    
                probs_wrong.append(1 - probs[correct_answer])

            probs_N.append(probs_wrong)

        probs_all_N.append(probs_N)

    # Aggrega tutti i dati
    probs_all_N = np.array(probs_all_N)
    probs_all_N = probs_all_N.transpose(1, 0, 2).reshape(len(list_N_wrongs), -1)

    # 🔥 Salva su file pickle
    save_path = f'data/data_{task}_{model_label}/majority_{condition}.pkl'
    with open(save_path, 'wb') as f:
        pickle.dump(probs_all_N, f)
    print(f'Saved {save_path}')

## SOCIAL IMPACT THEORY (IMMEDIACY) NATIONALITY

In [ ]:
import string
import os
import pickle
import re

labels = ['A', 'B']
list_N_wrongs = range(0, 11, 1)

folder = folder_general

nationalities = [
    "Italian",
    "French",
    "German",
    "American",
    "Brazilian",
    "Chinese",
    "Indian",
    "South African",
    "Russian",
    "Japanese"
]

conditions = ['same_nationality', 'different_nationality']

# Crea la directory "data" se non esiste
os.makedirs(f'data/data_{task}_{model_label}', exist_ok=True)

for condition in conditions:
    probs_all_N = []
    print(f'condition = {condition}')

    for correct_answer in ['A', 'B']:
        pattern = re.compile(rf"_({correct_answer})_p=.*\.png$")
        all_images = [f for f in os.listdir(folder) if pattern.search(f)]
        if not all_images:
            continue
        
        probs_N = []

        for N_wrong in list_N_wrongs:
            print(f'N_wrong = {N_wrong}')
            probs_wrong = []

            while len(probs_wrong) < n_simulations:
                image = random.choice(all_images)
                image_path = os.path.join(folder, image)

                # Genera le risposte dei confederati
                list_responses = []
                if N_wrong > 0:
                    for _ in range(N_wrong):
                        label = labels[1 - labels.index(correct_answer)]
                        response = get_neutral_responses(label, 1)[0]
                        list_responses.append(response)

                if condition == 'same_nationality':
                    nationality1 = random.choice(nationalities)
                    nationality2 = nationality1
                else:
                    nationality1, nationality2 = random.sample(nationalities, 2)
            
                # Crea il prompt
                if N_wrong == 0:
                    prompt = (
                        f"{prompt_base}"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )
                else:
                    prompt = (
                        f"Your nationality is {nationality1}.\n"
                        f"{prompt_base}"
                        f"The participants before you provided the following replies:\n"
                        + "\n".join(f"{nationality2}: reply {j+1}: {r}" for j, r in enumerate(list_responses)) + "\n"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )

                #inference
                if family == 'qwen':
                    probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'gemma':
                    probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'mistral':
                    probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
    
                probs_wrong.append(1 - probs[correct_answer])

            probs_N.append(probs_wrong)

        probs_all_N.append(probs_N)

    # Aggrega tutti i dati
    probs_all_N = np.array(probs_all_N)
    probs_all_N = probs_all_N.transpose(1, 0, 2).reshape(len(list_N_wrongs), -1)

    # 🔥 Salva su file pickle
    save_path = f'data/data_{task}_{model_label}/majority_{condition}.pkl'
    with open(save_path, 'wb') as f:
        pickle.dump(probs_all_N, f)
    print(f'Saved {save_path}')

## SOCIAL IMPACT THEORY (IMMEDIACY) GROUP

In [ ]:
import string
import os
import pickle
import re

labels = ['A', 'B']
list_N_wrongs = range(0, 11, 1)

folder = folder_general

groups = ['group 1', 'group 2']
conditions = ['same_group', 'different_group']

# Crea la directory "data" se non esiste
os.makedirs(f'data/data_{task}_{model_label}', exist_ok=True)

for condition in conditions:
    probs_all_N = []
    print(f'condition = {condition}')

    for correct_answer in ['A', 'B']:
        pattern = re.compile(rf"_({correct_answer})_p=.*\.png$")
        all_images = [f for f in os.listdir(folder) if pattern.search(f)]
        if not all_images:
            continue
        
        probs_N = []

        for N_wrong in list_N_wrongs:
            print(f'N_wrong = {N_wrong}')
            probs_wrong = []

            while len(probs_wrong) < n_simulations:
                image = random.choice(all_images)
                image_path = os.path.join(folder, image)

                # Genera le risposte dei confederati
                list_responses = []
                if N_wrong > 0:
                    for _ in range(N_wrong):
                        label = labels[1 - labels.index(correct_answer)]
                        response = get_neutral_responses(label, 1)[0]
                        list_responses.append(response)

                if condition == 'same_group':
                    group1 = random.choice(groups)
                    group2 = group1
                else:
                    shuffled_groups = groups.copy()
                    random.shuffle(shuffled_groups)
                    group1, group2 = shuffled_groups[0], shuffled_groups[1]
            
                # Crea il prompt
                if N_wrong == 0:
                    prompt = (
                        f"{prompt_base}"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )
                else:
                    prompt = (
                        f"You will be divided into groups with other participants. Your group is {group1}.\n"
                        f"{prompt_base}"
                        f"The participants before you provided the following replies:\n"
                        + "\n".join(f"{group2}: reply {j+1}: {r}" for j, r in enumerate(list_responses)) + "\n"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )

                #inference
                if family == 'qwen':
                    probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'gemma':
                    probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'mistral':
                    probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
    
                probs_wrong.append(1 - probs[correct_answer])

            probs_N.append(probs_wrong)

        probs_all_N.append(probs_N)

    # Aggrega tutti i dati
    probs_all_N = np.array(probs_all_N)
    probs_all_N = probs_all_N.transpose(1, 0, 2).reshape(len(list_N_wrongs), -1)

    # 🔥 Salva su file pickle
    save_path = f'data/data_{task}_{model_label}/majority_{condition}.pkl'
    with open(save_path, 'wb') as f:
        pickle.dump(probs_all_N, f)
    print(f'Saved {save_path}')

## SPATIAL PROXIMITY

In [ ]:
import string
import os
import pickle
import re


labels = ['A', 'B']
list_N_wrongs = range(0, 11, 1)

folder = folder_general

distances = [
    "here",
    "one meter away",
    "two meters away",
    "three meters away",
    "four meters away",
    "five meters away",
    "six meters away",
    "seven meters away",
    "eight meters away",
    "nine meters away",
    "ten meters away"
]

os.makedirs(f'data/data_{task}_{model_label}', exist_ok=True)

for distance in distances:
    probs_all_N = []
    print(f'distance = {distance}')

    for correct_answer in ['A', 'B']:
        pattern = re.compile(rf"_({correct_answer})_p=.*\.png$")
        all_images = [f for f in os.listdir(folder) if pattern.search(f)]
        if not all_images:
            continue
        
        probs_N = []

        for N_wrong in list_N_wrongs:
            print(f'N_wrong = {N_wrong}')
            probs_wrong = []

            while len(probs_wrong) < n_simulations:
                image = random.choice(all_images)
                image_path = os.path.join(folder, image)

                # Genera le risposte dei confederati
                list_responses = []
                if N_wrong > 0:
                    for _ in range(N_wrong):
                        label = labels[1 - labels.index(correct_answer)]
                        response = get_neutral_responses(label, 1)[0]
                        list_responses.append(response)
            
                # === Caso controllo (no social influence): prompt identico per tutti ===
                if N_wrong == 0:
                    prompt = (
                        f"{prompt_base}"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )
                else:
                    prompt = (
                        f"{prompt_base}"
                        f"The participants before you provided the following answers. Each participant is located at a specific distance from you, which is indicated before their response:\n"
                        + "\n".join(f"reply {j+1} (distance from you: {distance}): {r}" for j, r in enumerate(list_responses)) + "\n"
                        # + "\n".join(f"reply {j+1} ({distance} meters away): {r}" for j, r in enumerate(list_responses)) + "\n"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )
                
                #inference
                if family == 'qwen':
                    probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'gemma':
                    probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'mistral':
                    probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
    
                probs_wrong.append(1 - probs[correct_answer])

            probs_N.append(probs_wrong)

        probs_all_N.append(probs_N)

    # Aggrega tutti i dati
    probs_all_N = np.array(probs_all_N)

    # print(probs_all_N)
    probs_all_N = probs_all_N.transpose(1, 0, 2).reshape(len(list_N_wrongs), -1)

    # 🔥 Salva su file pickle
    save_path = f'data/data_{task}_{model_label}/majority_distance={distance}.pkl'
    with open(save_path, 'wb') as f:
        pickle.dump(probs_all_N, f)
    print(f'Saved {save_path}')

## TEMPORAL PROXIMITY

In [ ]:
import string
import os
import pickle
import re


labels = ['A', 'B']
list_N_wrongs = range(0, 11, 1)

folder = folder_general

scale = 'minute'

# Ridefinizione della variabile times
times = [
    "right now",
    f"2 {scale}s ago",
    f"4 {scale}s ago",
    f"6 {scale}s ago",
    f"8 {scale}s ago",
    f"10 {scale}s ago",
    f"12 {scale}s ago",
    f"14 {scale}s ago",
    f"16 {scale}s ago",
    f"18 {scale}s ago",
    f"20 {scale}s ago"
]

# Crea la directory "data" se non esiste
os.makedirs(f'data/data_{task}_{model_label}', exist_ok=True)

for time in times:
    probs_all_N = []
    print(f'time = {time}')

    for correct_answer in ['A', 'B']:
        pattern = re.compile(rf"_({correct_answer})_p=.*\.png$")
        all_images = [f for f in os.listdir(folder) if pattern.search(f)]
        if not all_images:
            continue
        
        probs_N = []

        for N_wrong in list_N_wrongs:
            print(f'N_wrong = {N_wrong}')
            probs_wrong = []

            while len(probs_wrong) < n_simulations:
                image = random.choice(all_images)
                image_path = os.path.join(folder, image)

                # Genera le risposte dei confederati
                list_responses = []
                if N_wrong > 0:
                    for _ in range(N_wrong):
                        label = labels[1 - labels.index(correct_answer)]
                        response = get_neutral_responses(label, 1)[0]
                        list_responses.append(response)
            
                # === Caso controllo (no social influence): prompt identico per tutti ===
                if N_wrong == 0:
                    prompt = (
                        f"{prompt_base}"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )
                else:
                    prompt = (
                        f"{prompt_base}"
                        f"The participants before you provided the following answers. Each participant responded at a different time, which is indicated before their response:\n"
                        + "\n".join(f"reply {j+1} (time of reply: {time}): {r}" for j, r in enumerate(list_responses)) + "\n"
                        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
                    )
                
                #inference
                if family == 'qwen':
                    probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'gemma':
                    probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
                if family == 'mistral':
                    probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
    
                probs_wrong.append(1 - probs[correct_answer])

            probs_N.append(probs_wrong)

        probs_all_N.append(probs_N)

    # Aggrega tutti i dati
    probs_all_N = np.array(probs_all_N)

    # print(probs_all_N)
    probs_all_N = probs_all_N.transpose(1, 0, 2).reshape(len(list_N_wrongs), -1)

    # 🔥 Salva su file pickle
    save_path = f'data/data_{task}_{model_label}/majority_time={time}.pkl'
    with open(save_path, 'wb') as f:
        pickle.dump(probs_all_N, f)
    print(f'Saved {save_path}')

## NORMATIVE EFFECTS

In [ ]:
variation_prompt_cant = [
    "Your response will not be shared with other participants.",
    "Other participants will not be able to see your answer.",
    "Your answer will not be visible to the group.",
    "No one in the group will see your response.",
    "Your response will not be displayed to others.",
    "The rest of the group will not view your answer.",
    "Your answer will not be shown to the other participants.",
    "The group will not have access to your response.",
    "Other participants will not know your answer.",
    "Your submission will not be visible to everyone.",
    "Your answer will be private.",
    "This response will not be shared with the group.",
    "The team will not see how you responded.",
    "No participant will view your answer.",
    "Your choice will not be shown to the others.",
    "The group will not receive your answer.",
    "Your response will not be visible to the rest of the team.",
    "Your answer will not be part of the group display.",
    "Other group members will not see your choice.",
    "This answer will not be shared with the team.",
    "Your input will not be visible to all participants.",
    "No one will see your response after submission.",
    "Your response will not be included in the group output.",
    "Other participants will not review your answer.",
    "This response will be confidential.",
    "Your answer will not be publicly visible in the group.",
    "The rest of the participants will not access your reply.",
    "Your response will not be visible in the shared results.",
    "No one else in the group will view your answer.",
    "Your reply will not be available to the group.",
    "No one else will see what you chose.",
    "Your selection will not be shared with the group.",
    "Others will not be able to check your response.",
    "Your reply will be hidden from the group.",
    "This input will not be shown to fellow participants.",
    "Others in the group will not be shown your answer.",
    "Your response will not appear in the group answers.",
    "No participant can see your submission.",
    "Your decision will not be shared with the others.",
    "The group will not have visibility of your reply.",
    "Your answer will not be shared post-submission.",
    "Other participants will not view this response.",
    "Your reply will not be made available to the group.",
    "This answer will not be shown to any participants.",
    "Others will not be able to see your input.",
    "Your response will not be seen by the team.",
    "This submission will not be visible to the group.",
    "No group member will see your answer.",
    "Your response will remain private.",
    "The team will not view your response.",
    "Your answer will not be visible to your teammates.",
    "Other members of the group will not view your response.",
    "Your response will not be seen by all participants.",
    "Others will not see how you answered.",
    "Your answer will not be reviewed by the group.",
    "Your input will not be visible after submission.",
    "The team will not be shown your answer.",
    "No group member can see your reply.",
    "Other participants will not access your answer.",
    "Your response will not be part of the shared results.",
    "Your answer will not be included in the visible group list.",
    "Group members will not see what you selected.",
    "Your choice will not be known to the team.",
    "This answer will not be included in the group responses.",
    "No one involved will see your input.",
    "Others will not receive your answer.",
    "Your answer will not be part of the public record.",
    "Your selection will not appear in the team output.",
    "The group will not know what you chose.",
    "Your response will be hidden from others.",
    "Your answer will not be included in the visible submissions.",
    "Other participants will not have access to your answer.",
    "Your reply will not be seen by the rest of the group.",
    "Others in the session will not view your response.",
    "Your answer will not be part of the collective results.",
    "Your answer will not be among those displayed to others.",
    "Other participants will not observe your answer.",
    "Your reply will not be distributed to all participants.",
    "Group members will not be shown your selection.",
    "The team will not receive your answer after submission.",
    "This input will not be shared with other group members.",
    "Your submission will not be added to the shared list.",
    "No one else will review your reply.",
    "Your response will not be open to the team.",
    "This answer will not be listed with the group responses.",
    "Others will not get access to your reply.",
    "Your selection will not be seen by other team members.",
    "This reply will not be visible to your peers.",
    "No one will receive your answer.",
    "Your decision will not be available to other participants.",
    "The group will not examine your answer.",
    "Your input will be kept private.",
    "Others in the group cannot inspect your reply.",
    "Your answer will not appear to others after you submit it.",
    "Your reply will not be added to the team's responses.",
    "This choice will not be accessible by the group.",
    "Other people in the session will not see your response.",
    "Your submission will not appear in the shared results.",
    "Your response will not be open for others to see.",
    "Your input will not be part of the group’s visible answers."
]

In [ ]:
variation_prompt_can = [
    "Your response will be shared with other participants.",
    "Other participants will be able to see your answer.",
    "Your answer will be visible to the group.",
    "Everyone in the group will see your response.",
    "Your response will be displayed to others.",
    "The rest of the group will view your answer.",
    "Your answer will be shown to the other participants.",
    "The group will have access to your response.",
    "Other participants will know your answer.",
    "Your submission will be visible to everyone.",
    "Your answer will not be private.",
    "This response will be shared with the group.",
    "The team will see how you responded.",
    "All participants will view your answer.",
    "Your choice will be shown to the others.",
    "The group will receive your answer.",
    "Your response will be visible to the rest of the team.",
    "Your answer will be part of the group display.",
    "Other group members will see your choice.",
    "This answer will be shared with the team.",
    "Your input will be visible to all participants.",
    "Everyone will see your response after submission.",
    "Your response will be included in the group output.",
    "Other participants will review your answer.",
    "This response will not be confidential.",
    "Your answer will be publicly visible in the group.",
    "The rest of the participants will access your reply.",
    "Your response will be visible in the shared results.",
    "All others in the group will view your answer.",
    "Your reply will be available to the group.",
    "Everyone else will see what you chose.",
    "Your selection will be shared with the group.",
    "Others will be able to check your response.",
    "Your reply will not be hidden from the group.",
    "This input will be shown to fellow participants.",
    "Others in the group will be shown your answer.",
    "Your response will appear in the group answers.",
    "All participants can see your submission.",
    "Your decision will be shared with the others.",
    "The group will have visibility of your reply.",
    "Your answer will be shared post-submission.",
    "Other participants will view this response.",
    "Your reply will be made available to the group.",
    "This answer will be shown to all participants.",
    "Others will be able to see your input.",
    "Your response will be seen by the team.",
    "This submission will be visible to the group.",
    "All group members will see your answer.",
    "Your response will not remain private.",
    "The team will view your response.",
    "Your answer will be visible to your teammates.",
    "Other members of the group will view your response.",
    "Your response will be seen by all participants.",
    "Others will see how you answered.",
    "Your answer will be reviewed by the group.",
    "Your input will be visible after submission.",
    "The team will be shown your answer.",
    "All group members can see your reply.",
    "Other participants will access your answer.",
    "Your response will be part of the shared results.",
    "Your answer will be included in the visible group list.",
    "Group members will see what you selected.",
    "Your choice will be known to the team.",
    "This answer will be included in the group responses.",
    "Everyone involved will see your input.",
    "Others will receive your answer.",
    "Your answer will be part of the public record.",
    "Your selection will appear in the team output.",
    "The group will know what you chose.",
    "Your response will not be hidden from others.",
    "Your answer will be included in the visible submissions.",
    "Other participants will have access to your answer.",
    "Your reply will be seen by the rest of the group.",
    "Others in the session will view your response.",
    "Your answer will be part of the collective results.",
    "Your answer will be among those displayed to others.",
    "Other participants will observe your answer.",
    "Your reply will be distributed to all participants.",
    "Group members will be shown your selection.",
    "The team will receive your answer after submission.",
    "This input will be shared with other group members.",
    "Your submission will be added to the shared list.",
    "All others will review your reply.",
    "Your response will be open to the team.",
    "This answer will be listed with the group responses.",
    "Others will get access to your reply.",
    "Your selection will be seen by other team members.",
    "This reply will be visible to your peers.",
    "Everyone will receive your answer.",
    "Your decision will be available to other participants.",
    "The group will examine your answer.",
    "Your input will not be kept private.",
    "Others in the group can inspect your reply.",
    "Your answer will appear to others after you submit it.",
    "Your reply will be added to the team's responses.",
    "This choice will be accessible by the group.",
    "Other people in the session will see your response.",
    "Your submission will appear in the shared results.",
    "Your response will be open for others to see.",
    "Your input will be part of the group’s visible answers."
]

### analyse public (can) setting

In [ ]:
import pickle
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

folder = folder_general

labels = ['A', 'B']
all_images = [f for f in os.listdir(folder) if f.endswith('.png')]

list_N_wrongs = range(0, 11, 1)

variation_prompt = variation_prompt_can

probs_correct_can = []

i = 0

for variation in variation_prompt:

    probs_correct = []
    i += 1
    print(f'variation {i}: {variation}')
    
    for N_wrong in list_N_wrongs:

        # 📥 Scegli immagine random
        image_name = random.choice(all_images)
        image_path = os.path.join(folder, image_name)
        image = Image.open(image_path)
        
        # 📥 Estrai correct_answer dinamicamente dal nome
        if "_A" in image_name:
            correct_answer = 'A'
            wrong_answer = 'B'
        elif "_B" in image_name:
            correct_answer = 'B'
            wrong_answer = 'A'
        else:
            raise ValueError(f"Formato filename non riconosciuto: {image_name}")

        # 📥 Genera risposte dei confederati (dando la risposta sbagliata!)
        list_responses = get_neutral_responses(wrong_answer, N_wrong)

        # Crea il prompt
        if N_wrong == 0:
            prompt = (
                f"{prompt_base}"
                f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
            )
        else:
            prompt = (
                f"{prompt_base}"
                f"{variation}\n"
                "The participants before you provided the following replies:\n"
                + "\n".join(f"reply {j+1}: {r}" for j, r in enumerate(list_responses)) + "\n"
                f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
            )
    
        # 📥 Query al modello

        #inference
        if family == 'qwen':
            probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'gemma':
            probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'mistral':
            probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        
        prob_correct = probs.get(correct_answer, 0.0)
        probs_correct.append(prob_correct)
    
    probs_correct_can.append(probs_correct)

# 🔥 Salva il risultato corretto
output_folder = f"data/data_{task}_{model_label}"
os.makedirs(output_folder, exist_ok=True)

save_path = os.path.join(output_folder, "normative_can.pkl")
with open(save_path, "wb") as f:
    pickle.dump(probs_correct_can, f)

print(f"✅ File salvato: {save_path}")

### analyse private (cant) setting

In [ ]:
import pickle
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

folder = folder_general

labels = ['A', 'B']
all_images = [f for f in os.listdir(folder) if f.endswith('.png')]

list_N_wrongs = range(0, 11, 1)

variation_prompt = variation_prompt_cant

probs_correct_cant = []

i = 0

for variation in variation_prompt:

    probs_correct = []
    i += 1
    print(f'variation {i}: {variation}')
    
    for N_wrong in list_N_wrongs:

        # 📥 Scegli immagine random
        image_name = random.choice(all_images)
        image_path = os.path.join(folder, image_name)
        image = Image.open(image_path)
        
        # 📥 Estrai correct_answer dinamicamente dal nome
        if "_A" in image_name:
            correct_answer = 'A'
            wrong_answer = 'B'
        elif "_B" in image_name:
            correct_answer = 'B'
            wrong_answer = 'A'
        else:
            raise ValueError(f"Formato filename non riconosciuto: {image_name}")

        # 📥 Genera risposte dei confederati (dando la risposta sbagliata!)
        list_responses = get_neutral_responses(wrong_answer, N_wrong)


        # Crea il prompt
        if N_wrong == 0:
            prompt = (
                f"{prompt_base}"
                f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
            )
        else:
            prompt = (
                f"{prompt_base}"
                f"{variation}\n"
                "The participants before you provided the following replies:\n"
                + "\n".join(f"reply {j+1}: {r}" for j, r in enumerate(list_responses)) + "\n"
                f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
            )
    
        # 📥 Query al modello

        #inference
        if family == 'qwen':
            probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'gemma':
            probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'mistral':
            probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        
        prob_correct = probs.get(correct_answer, 0.0)
        probs_correct.append(prob_correct)
    
    probs_correct_cant.append(probs_correct)

# 🔥 Salva il risultato corretto
output_folder = f"data/data_{task}_{model_label}"
os.makedirs(output_folder, exist_ok=True)

save_path = os.path.join(output_folder, "normative_cant.pkl")
with open(save_path, "wb") as f:
    pickle.dump(probs_correct_cant, f)

print(f"✅ File salvato: {save_path}")

## CHECK PERFORMANCES ON IMAGE POOL

In [ ]:
if task == 'asch_lines':
    
    import numpy as np
    import matplotlib.pyplot as plt
    import os
    
    # Parametri fissi
    reference_line = 300
    base_width = 300
    base_height = 600
    spacing = 120
    scale = 1.5
    dpi = 150
    labels = ['A', 'B']
    
    position = 0 
    correct_answer = labels[position]
        
    save_dir = f'images/images_lines_{model_label}'
    os.makedirs(save_dir, exist_ok=True)
    
    l_values = list(range(100, 500, 2))
    probs_1 = []
    scores_1 = []
    
    for l in l_values:
        print(f'l={l}')
        line2 = l
        filename = f"image_l_{l}.png"
        lengths = [reference_line, line2]
    
        image_path = os.path.join(save_dir, filename)
    
        
        generate_asch_image_two_lines(lengths, position, base_width, base_height, spacing, scale, image_path, dpi, labels)
        
        # Prompt al modello
        prompt = (
            f"In the image, there are three vertical lines labeled {labels[0]}, REFERENCE LINE, and {labels[1]}.\n"
            f"Which of the lines, {labels[0]} or {labels[1]}, is the same length as the REFERENCE LINE?\n"
            f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
        )
        
        # Chiamata al modello
        # Inference
        if family == 'qwen':
            probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'gemma':
            probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'mistral':
            probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
  
        # Salva la probabilità del token corretto ("A", poiché position=0)
        probs_1.append(probs[correct_answer])
        scores_1.append(output_scores[correct_answer])
    
        # Elimina l'immagine generata
        try:
            os.remove(image_path)
        except Exception as e:
            print(f"Errore nella cancellazione di {filename}: {e}")
    
    position = 1 
    correct_answer = labels[position]
            
    probs_2 = []
    scores_2 = []
    
    for l in l_values:
        print(f'l={l}')
        line2 = l
        filename = f"image_l_{l}.png"
        lengths = [reference_line, line2]
    
        image_path = os.path.join(save_dir, filename)
    
        
        generate_asch_image_two_lines(lengths, position, base_width, base_height, spacing, scale, image_path, dpi, labels)
        
        # Prompt al modello
        prompt = (
            f"In the image, there are three vertical lines labeled {labels[0]}, REFERENCE LINE, and {labels[1]}.\n"
            f"Which of the lines, {labels[0]} or {labels[1]}, is the same length as the REFERENCE LINE?\n"
            f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
        )
        
        # Inference
        if family == 'qwen':
            probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'gemma':
            probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'mistral':
            probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)

        # Salva la probabilità del token corretto ("A", poiché position=0)
        probs_2.append(probs[correct_answer])
        scores_2.append(output_scores[correct_answer])
    
        # Elimina l'immagine generata
        try:
            os.remove(image_path)
        except Exception as e:
            print(f"Errore nella cancellazione di {filename}: {e}")
    
    
    
    # Plot dei risultati
    plt.figure(figsize=(10, 6))
    plt.plot(l_values, probs_1, marker='o', color = 'blue')
    plt.plot(l_values, probs_2, marker='o', color = 'red')
    plt.xlabel("Lenght of other line", fontsize=12)
    plt.ylabel(f"Probability of Correct Token ({correct_answer})", fontsize=12)
    plt.title(f"{model_label}", fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    # Plot dei risultati
    plt.figure(figsize=(10, 6))
    plt.plot(l_values, scores_1, marker='o', color = 'blue')
    plt.plot(l_values, scores_2, marker='o', color = 'red')
    plt.xlabel("Lenght of other line", fontsize=12)
    plt.ylabel(f"Logit of Correct Token ({correct_answer})", fontsize=12)
    plt.title(f"{model_label}", fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    import pickle
    import numpy as np
    
    # Assicurati che siano array numpy
    l_values = np.array(l_values)
    probs_1 = np.array(probs_1)
    scores_1 = np.array(scores_1)
    
    # Salva in un dizionario
    data_1 = {
        'l_values': l_values,
        'probs': probs_1,
        'scores': scores_1
    }
    
    # Percorso di salvataggio
    save_path = f"data/data_{task}_{model_label}/check_images_{model_label}_A.pkl"
    
    # Salva con pickle
    with open(save_path, 'wb') as f:
        pickle.dump(data_1, f)
    
    print(f"\n✅ File salvato in {save_path}")
    
    
    # Assicurati che siano array numpy
    probs_2 = np.array(probs_2)
    scores_2 = np.array(scores_2)
    
    
    # Salva in un dizionario
    data_2 = {
        'l_values': l_values,
        'probs': probs_2,
        'scores': scores_2
    
    }
    
    # Percorso di salvataggio
    save_path = f"data/data_{task}_{model_label}/check_images_{model_label}_B.pkl"
    
    # Salva con pickle
    with open(save_path, 'wb') as f:
        pickle.dump(data_2, f)
    
    print(f"\n✅ File salvato in {save_path}")

In [ ]:
if task == 'color_recognition':

    import numpy as np
    import matplotlib.pyplot as plt
    import os
    
    # Parametri fissi
    reference_color = "(120,34,128)"
    base_rgb = (120, 34, 0)
    labels = ['A', 'B']
    save_dir = f'images/images_color_{model_label}'
    os.makedirs(save_dir, exist_ok=True)
    
    # Range del terzo valore RGB dell'other color
    z_values = list(range(0, 255, 1))
    
    position = 0
    correct_answer = labels[position]
    
    probs_1 = []
    scores_1 = []
    
    for z in z_values:
        print(f'z = {z}')
        other_color = f"({base_rgb[0]},{base_rgb[1]},{z})"
        filename = f"color_z_{z}.png"
        
        # Genera immagine
        create_image_color(
            reference_color=reference_color,
            other_color=other_color,
            position=position,
            directory_salvataggio=save_dir,
            nome_file=filename
        )
        
        # Prompt al modello
        image_path = os.path.join(save_dir, filename)
        prompt = (
            f"In the image, there are three colored squares labeled {labels[0]}, REFERENCE COLOR, and {labels[1]}.\n"
            f"Which of the squares, {labels[0]} or {labels[1]}, has the same color as the REFERENCE COLOR?\n"
            f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
        )
        
        # Inference
        if family == 'qwen':
            probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'gemma':
            probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'mistral':
            probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
   
        # Salva la probabilità del token corretto ("A", poiché position=0)
        probs_1.append(probs[correct_answer])
        scores_1.append(output_scores[correct_answer])
    
        # Elimina l'immagine generata
        try:
            os.remove(image_path)
        except Exception as e:
            print(f"Errore nella cancellazione di {filename}: {e}")
    
    position = 1
    correct_answer = labels[position]
    
    
    probs_2 = []
    scores_2 = []
    
    for z in z_values:
        print(f'z = {z}')
        other_color = f"({base_rgb[0]},{base_rgb[1]},{z})"
        filename = f"color_z_{z}.png"
        
        # Genera immagine
        create_image_color(
            reference_color=reference_color,
            other_color=other_color,
            position=position,
            directory_salvataggio=save_dir,
            nome_file=filename
        )
        
        # Prompt al modello
        image_path = os.path.join(save_dir, filename)
        prompt = (
            f"In the image, there are three colored squares labeled {labels[0]}, REFERENCE COLOR, and {labels[1]}.\n"
            f"Which of the squares, {labels[0]} or {labels[1]}, has the same color as the REFERENCE COLOR?\n"
            f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
        )
        
        # Inference
        if family == 'qwen':
            probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'gemma':
            probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'mistral':
            probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
  
        # Salva la probabilità del token corretto ("A", poiché position=0)
        probs_2.append(probs[correct_answer])
        scores_2.append(output_scores[correct_answer])
    
        # Elimina l'immagine generata
        try:
            os.remove(image_path)
        except Exception as e:
            print(f"Errore nella cancellazione di {filename}: {e}")
    
    # Plot dei risultati
    plt.plot(z_values, probs_1, marker='o', color = 'blue')
    plt.plot(z_values, probs_2, marker='o', color = 'red')
    plt.xlabel("Third RGB Value of Other Color (Blue channel)", fontsize=12)
    plt.ylabel(f"Probability of Correct Token ({correct_answer})", fontsize=12)
    plt.title(f"{model_label}", fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    
    import pickle
    import numpy as np
    
    # Assicurati che siano array numpy
    z_values = np.array(z_values)
    probs_1 = np.array(probs_1)
    scores_1 = np.array(scores_1)
    
    # Salva in un dizionario
    data_1 = {
        'z_values': z_values,
        'probs': probs_1,
        'scores': scores_1
    }
    
    # Percorso di salvataggio
    save_path = f"data/data_{task}_{model_label}/check_images_{model_label}_A.pkl"
    
    # Salva con pickle
    with open(save_path, 'wb') as f:
        pickle.dump(data_1, f)
    
    print(f"\n✅ File salvato in {save_path}")
    
    
    # Assicurati che siano array numpy
    probs_2 = np.array(probs_2)
    scores_2 = np.array(scores_2)
    
    
    # Salva in un dizionario
    data_2 = {
        'z_values': z_values,
        'probs': probs_2,
        'scores': scores_2
    
    }
    
    # Percorso di salvataggio
    save_path = f"data/data_{task}_{model_label}/check_images_{model_label}_B.pkl"
    
    # Salva con pickle
    with open(save_path, 'wb') as f:
        pickle.dump(data_2, f)
    
    print(f"\n✅ File salvato in {save_path}")


In [ ]:
if task == 'dots_estimation':

    import numpy as np
    import matplotlib.pyplot as plt
    import os
    
    # Parametri fissi
    points_reference = 10
    square_size = 300
    spacing = 100
    labels = ['A', 'B']
    save_dir = f'images/images_dots_{model_label}'
    os.makedirs(save_dir, exist_ok=True)
    
    # Range del terzo valore RGB dell'other color
    n_points_values = list(range(0, 100, 1))
    
    position = 0
    correct_answer = labels[position]
    
    probs_1 = []
    scores_1 = []
    
    
    for n_points in n_points_values:
        print(f'n_points = {n_points}')
        filename = f"dots_points_{n_points}.png"
        
        create_image_dots(points_reference, n_points, position, save_dir, filename)
        
        # Prompt al modello
        image_path = os.path.join(save_dir, filename)
        prompt = (
            f"In the image, there are three boxes labeled {labels[0]}, REFERENCE BOX, and {labels[1]}.\n"
            f"Which of the boxes, {labels[0]} or {labels[1]}, contains the same number of black dots as the REFERENCE BOX?\n"
            f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
        )
        
        # Inference
        if family == 'qwen':
            probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'gemma':
            probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'mistral':
            probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
 
        # Salva la probabilità del token corretto ("A", poiché position=0)
        probs_1.append(probs[correct_answer])
        scores_1.append(output_scores[correct_answer])
    
        # Elimina l'immagine generata
        try:
            os.remove(image_path)
        except Exception as e:
            print(f"Errore nella cancellazione di {filename}: {e}")
    
    position = 1
    correct_answer = labels[position]
    
    
    probs_2 = []
    scores_2 = []
    
    for n_points in n_points_values:
        print(f'n_points = {n_points}')
        filename = f"dots_points_{n_points}.png"
        
        create_image_dots(points_reference, n_points, position, save_dir, filename)
        
        # Prompt al modello
        image_path = os.path.join(save_dir, filename)
        prompt = (
            f"In the image, there are three boxes labeled {labels[0]}, REFERENCE BOX, and {labels[1]}.\n"
            f"Which of the boxes, {labels[0]} or {labels[1]}, contains the same number of black dots as the REFERENCE BOX?\n"
            f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
        )
        
        # Inference
        if family == 'qwen':
            probs, output_scores = single_query_qwen(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'gemma':
            probs, output_scores = single_query_gemma(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
        if family == 'mistral':
            probs, output_scores = single_query_mistral(prompt, image_path, labels, model, processor, generation_kwargs, print_flag = False)
   
        # Salva la probabilità del token corretto ("A", poiché position=0)
        probs_2.append(probs[correct_answer])
        scores_2.append(output_scores[correct_answer])
    
        # Elimina l'immagine generata
        try:
            os.remove(image_path)
        except Exception as e:
            print(f"Errore nella cancellazione di {filename}: {e}")
    
    # Plot dei risultati
    plt.plot(n_points_values, probs_1, marker='o', color = 'blue')
    plt.plot(n_points_values, probs_2, marker='o', color = 'red')
    plt.xlabel("n points", fontsize=12)
    plt.ylabel(f"Probability of Correct Token ({correct_answer})", fontsize=12)
    plt.title(f"{model_label}", fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    import pickle
    import numpy as np
    
    # Assicurati che siano array numpy
    n_points_values = np.array(n_points_values)
    probs_1 = np.array(probs_1)
    scores_1 = np.array(scores_1)
    
    # Salva in un dizionario
    data_1 = {
        'l_values': n_points_values,
        'probs': probs_1,
        'scores': scores_1
    }
    
    # Percorso di salvataggio
    save_path = f"data/data_{task}_{model_label}/check_images_{model_label}_A.pkl"
    
    # Salva con pickle
    with open(save_path, 'wb') as f:
        pickle.dump(data_1, f)
    
    print(f"\n✅ File salvato in {save_path}")
    
    
    # Assicurati che siano array numpy
    probs_2 = np.array(probs_2)
    scores_2 = np.array(scores_2)
    
    
    # Salva in un dizionario
    data_2 = {
        'l_values': n_points_values,
        'probs': probs_2,
        'scores': scores_2
    
    }
    
    # Percorso di salvataggio
    save_path = f"data/data_{task}_{model_label}/check_images_{model_label}_B.pkl"
    
    # Salva con pickle
    with open(save_path, 'wb') as f:
        pickle.dump(data_2, f)
    
    print(f"\n✅ File salvato in {save_path}")